# Learning Rate Scheduling — Complete Theory

---

## The Problem With a Fixed Learning Rate

Right now your optimizer uses `lr=0.0001` forever. Think of it like hiking to a valley (the loss minimum):

- **Early training:** You're far from the bottom, big steps are fine and fast
- **Late training:** You're near the bottom, big steps cause you to overshoot and bounce around

A fixed LR is a compromise — small enough to not overshoot late, but that means unnecessarily slow early on. Schedulers solve this by **automatically adjusting LR during training.**

---

## What a Scheduler Actually Does

A scheduler wraps your optimizer and modifies its `lr` parameter according to a rule. Every time you call `scheduler.step()`, it looks at the current state and decides whether to change the LR.

The scheduler doesn't replace the optimizer — it sits on top of it:

```
optimizer → controls weight updates
scheduler → controls the optimizer's learning rate
```

---

## The Three Schedulers Worth Knowing

### 1. StepLR
Reduces LR by a fixed factor every N epochs. Simple and predictable.

```
Epoch 1-10:  lr = 0.001
Epoch 11-20: lr = 0.0001   ← multiplied by 0.1
Epoch 21-30: lr = 0.00001  ← multiplied by 0.1 again
```

**Parameters:**
- `step_size` — how many epochs between reductions
- `gamma` — multiplication factor (0.1 means divide by 10)

**Import and syntax:**
```python
from torch.optim.lr_scheduler import StepLR
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)
```

---

### 2. ReduceLROnPlateau
The smart one. Instead of reducing on a schedule, it watches your val loss and reduces LR only when progress stalls. "Plateau" means the loss stopped improving.

```
Val loss improving → keep LR
Val loss stuck for N epochs → reduce LR automatically
```

**Parameters:**
- `mode` — `'min'` if watching loss (lower is better), `'max'` if watching accuracy
- `patience` — how many epochs of no improvement before reducing
- `factor` — multiplication factor (like gamma above)
- `min_lr` — floor so LR never goes to zero

**Import and syntax:**
```python
from torch.optim.lr_scheduler import ReduceLROnPlateau
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, min_lr=1e-6)
```

**Critical difference:** This scheduler needs to know the val loss to make decisions. So you call it differently — you pass the metric in:
```python
scheduler.step(val_loss)  # not just scheduler.step()
```

---

### 3. CosineAnnealingLR
Reduces LR following a cosine curve — starts high, smoothly decays to near zero, optionally restarts. Popular in modern deep learning.

```
LR
│\
│  \
│    \___
│         \___
└──────────────── epochs
```

**Parameters:**
- `T_max` — number of epochs for one full decay cycle
- `eta_min` — minimum LR at the bottom of the curve

**Import and syntax:**
```python
from torch.optim.lr_scheduler import CosineAnnealingLR
scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)
```

---

## Where `scheduler.step()` Goes

This is the most common mistake. **For all schedulers except ReduceLROnPlateau**, you call `scheduler.step()` once per epoch, *after* the full training loop:

```
for epoch in range(num_epochs):
    # full training loop
    # full validation loop
    scheduler.step()        ← once per epoch, after both loops
```

**For ReduceLROnPlateau** specifically:
```
scheduler.step(avg_val_loss)  ← pass the metric it's watching
```

---

## How to Monitor the LR

You can check the current LR at any time:
```python
current_lr = optimizer.param_groups[0]['lr']
print(f"Current LR: {current_lr}")
```

Print this each epoch so you can actually see the scheduler working.

---

## Which One For Your Project?

**ReduceLROnPlateau is the right choice for you right now.** Here's why — StepLR and Cosine require you to know upfront how many epochs you'll train. ReduceLROnPlateau adapts to your actual training dynamics. Since your model is simple and your dataset is small, it will plateau relatively quickly and the scheduler will respond automatically.


## TODOs — Learning Rate Scheduling

---

### TODO 1: Add ReduceLROnPlateau to Your Existing Loop

**Goal:** Plug a scheduler into your current training loop and observe it affecting the LR.

**Requirements:**
- Import `ReduceLROnPlateau` from `torch.optim.lr_scheduler`
- Create the scheduler with `mode='min'`, `patience=2`, `factor=0.5`, `min_lr=1e-6`
- Call `scheduler.step()` in the right place with the right argument
- Print current LR at the end of each epoch
- Run for 10 epochs and show me the full output

**Expected output format:**
```
--- Epoch 1/10 Summary ---
Train Loss: 0.8273 | Train Acc: 51.99%
Val Accuracy: 58.12%
Current LR: 0.0001

--- Epoch 5/10 Summary ---
...
Current LR: 0.00005  ← you should see this change if val loss plateaus
```

**Hints:**
- Scheduler is created after the optimizer, takes optimizer as first argument
- You're watching val loss → `mode='min'`
- You still need to compute `avg_val_loss` in your val loop (you noticed this was missing earlier)
- `scheduler.step()` goes after both training and validation loops complete
- LR check: `optimizer.param_groups[0]['lr']`

---

### TODO 2: Experiment With Patience

**Goal:** Understand how `patience` controls scheduler behavior.

After TODO 1 works — change `patience` to `0` and rerun 10 epochs. Then change to `5` and rerun.

**Questions to answer from observation:**
- With `patience=0`, when does LR first drop?
- With `patience=5`, does LR drop at all in 10 epochs?
- What does this tell you about choosing patience for a real project?

No code to write here — just run and observe.

---

Show me TODO 1 output first. Don't move to TODO 2 until the LR is visibly printing each epoch.

In [20]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from pathlib import Path
import os
import sys
import json

# Check if __file__ exists (it won't in Jupyter)
try:
    current_dir = Path(__file__).parent
except NameError:
    # If in Jupyter, use the current working directory
    current_dir = Path(os.getcwd())

# Add project root to Python path
project_root = current_dir.parent.parent.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.audio_utils import load_audio


# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")


✅ Using Apple Silicon GPU
PyTorch version: 2.10.0
Device: mps


### loading the sample

In [21]:
class SpeakerDataset(torch.utils.data.Dataset):
    def __init__(self, manifest_path):
        # Load manifest JSON
        # Store all entries as self.data
        with open(manifest_path, "r") as f:
            self.data = json.load(f)
    
    def __len__(self):
        # Return number of samples
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get entry at index idx
        # Load audio from disk
        # Extract features
        # Get label (num_speakers - 1)
        # Return (features, label)
        entry = self.data[idx]
        mixture_path = Path(entry['mixture_path'])
        mixture_audio, _ = load_audio(mixture_path)
        
        mixture_tensor = torch.from_numpy(mixture_audio)
        mixture_tensor = mixture_tensor
        feature_tensor = self.feature_extraction(mixture_tensor)
        
        speaker_count = int(entry['num_speakers']) - 1
        label_tensor = torch.tensor(speaker_count, dtype=torch.long)
        
        return feature_tensor, label_tensor
    
    def feature_extraction(
            self,
            audio: torch.Tensor
    ) -> torch.Tensor:
        mean = torch.mean(audio)
        std = torch.std(audio)
        min = torch.min(audio)
        max = torch.max(audio)
        square = audio ** 2
        energy = torch.mean(square)
        
        return torch.stack([mean, std, min, max, energy])

### getting the path to the dataset and loading the metadata

In [22]:
train_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "train" / "train_manifest.json"
train_dataset = SpeakerDataset(train_manifest_path)
print(len(train_dataset))

val_manifest_path = Path.cwd().parent.parent.parent / "data" / "processed" / "val" / "val_manifest.json"
val_dataset = SpeakerDataset(val_manifest_path)
print(len(val_dataset))


10000
10000


In [23]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

### creating the neural network

In [24]:
class SpeakerCounter(nn.Module):
    """
    Classifier that predicts number of speakers (1, 2, or 3)
    from audio features.
    """
    def __init__(self, input_size=5, hidden_sizes=[64, 32, 16], num_classes=3):
        super().__init__()
        # TODO: Build network with given architecture
        # Hint: Use nn.Sequential or define layers individually
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_sizes[0]),
            nn.BatchNorm1d(hidden_sizes[0]),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_sizes[0],hidden_sizes[1]),
            nn.BatchNorm1d(hidden_sizes[1]),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_sizes[1],hidden_sizes[2]),
            nn.BatchNorm1d(hidden_sizes[2]),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_sizes[2],num_classes)
        )
        
    def forward(self, x):
        # TODO: Implement forward pass
        return self.network(x)

### calculating the loss

In [25]:
def compute_accuracy(predictions, labels):
    """
    predictions: (batch_size, 3) logits
    labels: (batch_size,) true classes
    
    Returns: accuracy as percentage
    """
    predicted_classes = torch.argmax(predictions, dim=1)
    correct = (predicted_classes == labels).sum().item()
    total = labels.size(0)
    return 100.0 * correct / total

### training and validation loop

### todo 1

In [26]:
model = SpeakerCounter().to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5, min_lr=1e-6)
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0
    running_train_acc = 0.0
    steps_per_epoch = len(train_loader)
    
    for step, batch in enumerate(train_loader):
        features, labels = batch[0].to(device), batch[1].to(device)
        
        # Forward pass
        logits = model(features)  
        
        # Compute loss
        loss = F.cross_entropy(logits, labels)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer.step()
        optimizer.zero_grad()
        
        # Compute accuracy
        running_train_loss += loss.item() 
        running_train_acc += compute_accuracy(logits, labels)
        
        # Optional: Print batch progress every 50 batches within the epoch
        if (step + 1) % 50 == 0:
            print(f"  train Batch {step+1}/{steps_per_epoch} | Loss: {loss.item():.4f}")
    
    # 3. Print Summary at the end of each FULL epoch
    avg_train_loss = running_train_loss / steps_per_epoch
    avg_train_acc = running_train_acc / steps_per_epoch    
    
    model.eval()
    running_val_loss = 0.0
    running_val_acc = 0.0      
    
    with torch.no_grad():
        for steps, batch in enumerate(val_loader):
            
            features, labels = batch[0].to(device), batch[1].to(device)
            
            # Forward pass
            logits = model(features)  
            
            # Compute loss
            loss = F.cross_entropy(logits, labels)
            
            # number of correct class
            running_val_loss += loss.item()
            running_val_acc += compute_accuracy(logits, labels)  
            
            # Optional: Print batch progress every 50 batches within the epoch
            if (steps + 1) % 50 == 0:
                print(f"  val Batch {steps+1}/{steps_per_epoch} | Loss: {loss.item():.4f}")
    
    # Calculate Averages
    avg_val_loss = running_val_loss / len(val_loader)
    avg_val_acc = running_val_acc / len(val_loader)    
    
    # --- LOGGING & SCHEDULER ---
    print(f"\n--- Epoch {epoch+1}/{num_epochs} Summary ---")
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.2f}%")
    print(f"Val Loss:   {avg_val_loss:.4f} | Val Acc:   {avg_val_acc:.2f}%")
    
    scheduler.step(avg_val_loss)
    
    # Optional: Print current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr}\n")




  train Batch 50/156 | Loss: 1.0870
  train Batch 100/156 | Loss: 1.0522
  train Batch 150/156 | Loss: 0.9692
  val Batch 50/156 | Loss: 1.0487
  val Batch 100/156 | Loss: 1.0255
  val Batch 150/156 | Loss: 1.0305

--- Epoch 1/5 Summary ---
Train Loss: 1.0243 | Train Acc: 48.77%
Val Loss:   1.0232 | Val Acc:   50.06%
Current Learning Rate: 0.0001

  train Batch 50/156 | Loss: 0.9660
  train Batch 100/156 | Loss: 0.9920
  train Batch 150/156 | Loss: 0.9750
  val Batch 50/156 | Loss: 0.9879
  val Batch 100/156 | Loss: 0.9732
  val Batch 150/156 | Loss: 0.9752

--- Epoch 2/5 Summary ---
Train Loss: 0.9723 | Train Acc: 49.94%
Val Loss:   0.9666 | Val Acc:   49.97%
Current Learning Rate: 0.0001

  train Batch 50/156 | Loss: 0.9319
  train Batch 100/156 | Loss: 0.9139
  train Batch 150/156 | Loss: 0.8810
  val Batch 50/156 | Loss: 0.9396
  val Batch 100/156 | Loss: 0.9239
  val Batch 150/156 | Loss: 0.9337

--- Epoch 3/5 Summary ---
Train Loss: 0.9266 | Train Acc: 50.79%
Val Loss:   0.9188 |

### todo 2

In [27]:
model = SpeakerCounter().to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=0, factor=0.5, min_lr=1e-6)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0
    running_train_acc = 0.0
    steps_per_epoch = len(train_loader)
    
    for step, batch in enumerate(train_loader):
        features, labels = batch[0].to(device), batch[1].to(device)
        
        # Forward pass
        logits = model(features)  
        
        # Compute loss
        loss = F.cross_entropy(logits, labels)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer.step()
        optimizer.zero_grad()
        
        # Compute accuracy
        running_train_loss += loss.item() 
        running_train_acc += compute_accuracy(logits, labels)
        
        # Optional: Print batch progress every 50 batches within the epoch
        if (step + 1) % 50 == 0:
            print(f"  train Batch {step+1}/{steps_per_epoch} | Loss: {loss.item():.4f}")
    
    # 3. Print Summary at the end of each FULL epoch
    avg_train_loss = running_train_loss / steps_per_epoch
    avg_train_acc = running_train_acc / steps_per_epoch    
    
    model.eval()
    running_val_loss = 0.0
    running_val_acc = 0.0      
    
    with torch.no_grad():
        for steps, batch in enumerate(val_loader):
            
            features, labels = batch[0].to(device), batch[1].to(device)
            
            # Forward pass
            logits = model(features)  
            
            # Compute loss
            loss = F.cross_entropy(logits, labels)
            
            # number of correct class
            running_val_loss += loss.item()
            running_val_acc += compute_accuracy(logits, labels)  
            
            # Optional: Print batch progress every 50 batches within the epoch
            if (steps + 1) % 50 == 0:
                print(f"  val Batch {steps+1}/{steps_per_epoch} | Loss: {loss.item():.4f}")
    
    # Calculate Averages
    avg_val_loss = running_val_loss / len(val_loader)
    avg_val_acc = running_val_acc / len(val_loader)    
    
    # --- LOGGING & SCHEDULER ---
    print(f"\n--- Epoch {epoch+1}/{num_epochs} Summary ---")
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.2f}%")
    print(f"Val Loss:   {avg_val_loss:.4f} | Val Acc:   {avg_val_acc:.2f}%")
    
    scheduler.step(avg_val_loss)
    
    # Optional: Print current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr}\n")




  train Batch 50/156 | Loss: 1.0909
  train Batch 100/156 | Loss: 1.0924
  train Batch 150/156 | Loss: 1.0204
  val Batch 50/156 | Loss: 0.9688
  val Batch 100/156 | Loss: 1.0146
  val Batch 150/156 | Loss: 0.9981

--- Epoch 1/10 Summary ---
Train Loss: 1.0876 | Train Acc: 49.10%
Val Loss:   1.0102 | Val Acc:   49.90%
Current Learning Rate: 0.0001

  train Batch 50/156 | Loss: 0.9913
  train Batch 100/156 | Loss: 1.0994
  train Batch 150/156 | Loss: 1.0534
  val Batch 50/156 | Loss: 0.9417
  val Batch 100/156 | Loss: 0.9903
  val Batch 150/156 | Loss: 0.9717

--- Epoch 2/10 Summary ---
Train Loss: 1.0218 | Train Acc: 50.29%
Val Loss:   0.9814 | Val Acc:   49.90%
Current Learning Rate: 0.0001

  train Batch 50/156 | Loss: 0.9784
  train Batch 100/156 | Loss: 0.9682
  train Batch 150/156 | Loss: 0.9671
  val Batch 50/156 | Loss: 0.9121
  val Batch 100/156 | Loss: 0.9576
  val Batch 150/156 | Loss: 0.9400

--- Epoch 3/10 Summary ---
Train Loss: 0.9741 | Train Acc: 51.00%
Val Loss:   0.948

In [28]:
model = SpeakerCounter().to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5, min_lr=1e-6)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0
    running_train_acc = 0.0
    steps_per_epoch = len(train_loader)
    
    for step, batch in enumerate(train_loader):
        features, labels = batch[0].to(device), batch[1].to(device)
        
        # Forward pass
        logits = model(features)  
        
        # Compute loss
        loss = F.cross_entropy(logits, labels)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer.step()
        optimizer.zero_grad()
        
        # Compute accuracy
        running_train_loss += loss.item() 
        running_train_acc += compute_accuracy(logits, labels)
        
        # Optional: Print batch progress every 50 batches within the epoch
        if (step + 1) % 50 == 0:
            print(f"  train Batch {step+1}/{steps_per_epoch} | Loss: {loss.item():.4f}")
    
    # 3. Print Summary at the end of each FULL epoch
    avg_train_loss = running_train_loss / steps_per_epoch
    avg_train_acc = running_train_acc / steps_per_epoch    
    
    model.eval()
    running_val_loss = 0.0
    running_val_acc = 0.0      
    
    with torch.no_grad():
        for steps, batch in enumerate(val_loader):
            
            features, labels = batch[0].to(device), batch[1].to(device)
            
            # Forward pass
            logits = model(features)  
            
            # Compute loss
            loss = F.cross_entropy(logits, labels)
            
            # number of correct class
            running_val_loss += loss.item()
            running_val_acc += compute_accuracy(logits, labels)  
            
            # Optional: Print batch progress every 50 batches within the epoch
            if (steps + 1) % 50 == 0:
                print(f"  val Batch {steps+1}/{steps_per_epoch} | Loss: {loss.item():.4f}")
    
    # Calculate Averages
    avg_val_loss = running_val_loss / len(val_loader)
    avg_val_acc = running_val_acc / len(val_loader)    
    
    # --- LOGGING & SCHEDULER ---
    print(f"\n--- Epoch {epoch+1}/{num_epochs} Summary ---")
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.2f}%")
    print(f"Val Loss:   {avg_val_loss:.4f} | Val Acc:   {avg_val_acc:.2f}%")
    
    scheduler.step(avg_val_loss)
    
    # Optional: Print current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr}\n")




  train Batch 50/156 | Loss: 1.1715
  train Batch 100/156 | Loss: 1.2123
  train Batch 150/156 | Loss: 1.1838
  val Batch 50/156 | Loss: 1.0677
  val Batch 100/156 | Loss: 1.1021
  val Batch 150/156 | Loss: 1.0736

--- Epoch 1/10 Summary ---
Train Loss: 1.1687 | Train Acc: 35.88%
Val Loss:   1.0886 | Val Acc:   40.63%
Current Learning Rate: 0.0001

  train Batch 50/156 | Loss: 1.1386
  train Batch 100/156 | Loss: 1.0739
  train Batch 150/156 | Loss: 1.0528
  val Batch 50/156 | Loss: 1.0158
  val Batch 100/156 | Loss: 1.0501
  val Batch 150/156 | Loss: 1.0331

--- Epoch 2/10 Summary ---
Train Loss: 1.0892 | Train Acc: 44.37%
Val Loss:   1.0406 | Val Acc:   49.89%
Current Learning Rate: 0.0001

  train Batch 50/156 | Loss: 1.1031
  train Batch 100/156 | Loss: 1.0112
  train Batch 150/156 | Loss: 1.0492
  val Batch 50/156 | Loss: 0.9767
  val Batch 100/156 | Loss: 1.0128
  val Batch 150/156 | Loss: 1.0004

--- Epoch 3/10 Summary ---
Train Loss: 1.0323 | Train Acc: 48.96%
Val Loss:   1.004